In [ ]:
import sys
import pandas as pd
sys.path.append('/Users/BKieft/Metabolomics/metatlas2')
import metatlas2.database_interact as dbi
import metatlas2.pubchem_retrieval as pcr
import metatlas2.load_tools as ldt
import metatlas2.lcmsruns_tools as lrt
import metatlas2.ms1_ms2_analysis as msa
import metatlas2.rt_align_tools as rat
import metatlas2.targeted_analysis as tga
pd.options.display.max_colwidth = 300

In [ ]:
# Required: QC (model) and Target (reference) atlas for performing RT alignment
QC_ATLAS_UID = "atlas-be09322758f046a18a8166ddf7a00cbc"
TARGET_ATLAS_UID = "atlas-89b3d74932b94b1aa01b01bca7668c11"
# Required: Project name
PROJECT_NAME = "20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519"
# Required: load config file
CONFIG = ldt.load_metatlas_config("/Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml")
# Required: paths based on config
PROJECT_FILES_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/lcmsruns"
PROJECT_DB_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/{PROJECT_NAME}.duckdb"

In [ ]:
dbi.create_project_database(PROJECT_DB_PATH, CONFIG)

In [ ]:
lcmsrun_files = dbi.save_lcmsruns_to_db(PROJECT_DB_PATH, PROJECT_NAME, PROJECT_FILES_PATH, CONFIG)

In [ ]:
matches_df, matching_stats = msa.extract_and_match_qc_compounds(PROJECT_DB_PATH, QC_ATLAS_UID, CONFIG)

In [ ]:
best_model, model_results, model_stats = rat.build_rt_alignment_model(matches_df, CONFIG)

In [ ]:
rt_summary, rt_atlas_uid = rat.apply_rt_correction_to_target(PROJECT_DB_PATH, TARGET_ATLAS_UID, CONFIG, best_model, lcmsrun_files, model_results)

In [ ]:
dbi.get_rt_correction_table_entry(PROJECT_DB_PATH, rt_atlas_uid)

In [ ]:
dbi.validate_database(CONFIG, PROJECT_DB_PATH)

In [ ]:
ANALYSIS_ATLAS_UID = rt_atlas_uid
atlas_df_ft, eics, ms2_data_with_hits, ms2_hits = tga.run_targeted_analysis_workflow(PROJECT_DB_PATH, ANALYSIS_ATLAS_UID, CONFIG)

In [ ]:
msa.summarize_ms2_hits(ms2_hits, CONFIG)

In [ ]:
plot_data = tga.set_up_plot_data(eics, atlas_df_ft, ms2_data_with_hits)

In [ ]:
gui = tga.create_gui(plot_data, atlas_df_ft, CONFIG)

In [ ]:
display(gui)

In [ ]:
plot_data['AGPKZVBTJJNPAG-WHFBIAKZSA-N']['new_atlas_data']